# Thư viện

In [1]:
%pip install -qq  transformers underthesea sentencepiece
%pip install -qq torch
%pip install -qq rouge-score sacrebleu
%pip show torch

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Name: torch
Version: 2.13.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License-Expression: Apache-2.0 AND Apache-2.0 WITH LLVM-exception AND BSD-2-Clause AND BSD-3-Clause AND BSL-1.0 AND MIT
Location: /home/dungcony/.pyenv/versions/3.13.14/lib/python3.13/site-packages
Requires: cuda-bindings, cuda-toolkit, filelock, fsspec, jinja2, networkx, nvidia-cudnn-cu13, nvidia-cusparselt-cu13, nvidia-nccl-cu13, nvidia-nvshmem-cu13, setuptools, sympy, triton, typing-extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import torch
# from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
from sklearn.cluster import KMeans
from rouge_score import rouge_scorer
import sacrebleu
from underthesea import sent_tokenize, word_tokenize

# Tải mô hình

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
# model PhoBert

model = AutoModel.from_pretrained("vinai/phobert-base").to(device)
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 30967.48it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(64001, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(258, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=Tru

# Lấy embedding câu

In [5]:
# Chuyển từ câu -> vectors embedding 
def get_embeddings(sentences, batch_size=32):
    embeddings  = []
    
    # Tắt tính toán đạo hàm (gradient) để chạy nhanh và tiết kiệm RAM
    with torch.no_grad():
        # Lặp qua từng mẻ (batch), mỗi mẻ xử lý 32 câu cùng lúc
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i+batch_size]
            
            # Tokenizer: Đổi chữ thành ID số. padding=True chèn thêm từ rỗng để các câu bằng nhau
            inputs = tokenizer(batch, padding=True, return_tensors="pt", truncation=True, max_length=256).to(device)
            
            # Đưa ma trận ID số vào mô hình PhoBERT
            outputs = model(**inputs)

            # Lấy vector biểu diễn của từng chữ (token)
            last_hidden = outputs.last_hidden_state   
            
            # Lấy màng lọc attention (chữ thật = 1, chữ độn = 0)
            mask = inputs["attention_mask"].unsqueeze(-1)   
            
            # Triệt tiêu nhiễu của chữ độn bằng cách nhân với 0
            masked_hidden = last_hidden * mask 
            
            # Cộng gộp vector của tất cả chữ thật trong câu
            sum_hidden = masked_hidden.sum(dim=1)
            
            # Đếm xem có bao nhiêu chữ thật trong câu
            lengths = mask.sum(dim=1)  
            
            # Chia trung bình cộng (Mean Pooling) -> Ra được 1 vector duy nhất cho cả câu
            mean_pooled = sum_hidden / lengths        

            # Lưu vào mảng và đẩy từ GPU về CPU
            embeddings.append(mean_pooled.cpu().numpy())

    # Ghép lại thành 1 ma trận duy nhất cho toàn bộ văn bản
    return np.vstack(embeddings)

# Tóm tắt extractive bằng K-Means

In [6]:
# K-Means: Phân cụm các câu liên quan vào chung nhóm
# Từ mỗi nhóm lấy ra 1 câu gần nhất tâm cụm làm câu tóm tắt. Số tâm cụm = số câu tóm tắt

def kmeans_summary(sentences):
    # Số cụm (số lượng câu tóm tắt) = Căn bậc 2 của tổng số câu gốc
    k = int(np.ceil(len(sentences)**0.5))    
    
    # Lấy vector đại diện cho tất cả các câu
    emb = get_embeddings(sentences)     

    # Khởi tạo và chạy thuật toán K-Means gom cụm
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(emb)

    summary = []
    # Duyệt qua từng cụm
    for cid in range(k):
        # Lấy danh sách vị trí của các câu thuộc cụm hiện tại
        idx = np.where(kmeans.labels_ == cid)[0]     
        
        # Tọa độ Tâm của cụm
        centroid = kmeans.cluster_centers_[cid]                               
        
        # Tính khoảng cách từ các câu đến Tâm cụm -> Chọn câu GẦN TÂM NHẤT (Khoảng cách Euclid nhỏ nhất)
        best = idx[np.argmin(np.linalg.norm(emb[idx] - centroid, axis=1))]   
        
        # Lưu câu được chọn và vị trí gốc của nó
        summary.append((best, sentences[best]))

    # Sắp xếp lại các câu tóm tắt theo đúng thứ tự xuất hiện gốc trong bài báo
    summary = sorted(summary)                                                 
    
    # Ghép các câu lại thành văn bản
    return " ".join([s for _, s in summary])

# Tóm tắt extractive bằng DBSCAN

In [7]:
#dbscan: phân cụm dựa trên mật độ
# ý tưởng: 1 điểm thuộc 1 cụm nếu nó nawmgf trong vùng có mật độ cao, có ít nhất min_samples điểm khác 

from sklearn.cluster import DBSCAN

def dbscan_summary(sentences, eps=3.7, min_samples=2):    #eps: ngưỡng khoảng cách 2 điểm coi là gần nhau

    emb = get_embeddings(sentences)
    #tính phân bố kcachs
    
    db = DBSCAN(eps=eps, min_samples=min_samples, metric="euclidean")
    labels = db.fit_predict(emb)

    unique_clusters = sorted([c for c in set(labels) if c != -1])  # bỏ cụm nhiễu -1

    #vẫn là chọn câu từ các cụm
    summary = []
    for cid in unique_clusters:
        idx = np.where(labels == cid)[0]
        centroid = emb[idx].mean(axis=0)
        best = idx[np.argmin(np.linalg.norm(emb[idx] - centroid, axis=1))]
        summary.append((best, sentences[best]))

    summary = sorted(summary)
    return " ".join([s for _, s in summary])

# Đánh giá (ROUGE-1, ROUGE-2, ROUGE-L, BLEU)

In [8]:
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)

def evaluate(gold, pred):
    gold = " ".join(word_tokenize(gold))   #tách từ theo tiếng việt rồi ghép lại, vì tiếng việt có từ ghép "học sinh" != "học", "sinh"
    pred = " ".join(word_tokenize(pred))
    
    r = rouge.score(gold, pred)
    bleu = sacrebleu.corpus_bleu([pred], [[gold]]).score

    return {
        "rouge1": round(r["rouge1"].fmeasure, 4),
        "rouge2": round(r["rouge2"].fmeasure, 4),
        "rougeL": round(r["rougeL"].fmeasure, 4),
        
        "bleu": round(bleu / 100, 4)  # sacrebleu ở thang đo 0-100 -> 0-1
    }

# Thử với dữ liệu

In [9]:
#sample: text là bản gốc, gold là bản tóm tắt (gold trong Gold Summary Reference)

text = """Sáng nay, Ủy ban Nhân dân thành phố công bố kế hoạch phát triển giao thông công cộng giai đoạn 2025–2030.
Theo đó, thành phố sẽ đầu tư hơn 5.000 tỷ đồng để mở rộng mạng lưới xe buýt và xây dựng thêm ba tuyến BRT mới.
Đại diện Sở Giao thông cho biết mục tiêu là tăng tỷ lệ người dân sử dụng phương tiện công cộng lên 25%.
Ngoài ra, thành phố cũng sẽ triển khai hệ thống vé điện tử tích hợp cho tất cả các loại hình vận tải.
Việc áp dụng vé điện tử được kỳ vọng sẽ giúp giảm thời gian lên xuống xe và hạn chế gian lận.
Bên cạnh đó, thành phố đặt mục tiêu giảm 15% lượng xe cá nhân lưu thông trong khu vực trung tâm.
Các chuyên gia cho rằng muốn đạt mục tiêu này, cần đồng thời tăng phí đỗ xe và kiểm soát chặt xe máy cũ.
Thành phố cũng sẽ thí điểm làn đường riêng cho xe buýt trên ba trục chính từ quý II năm sau.
Người dân bày tỏ kỳ vọng hệ thống mới sẽ giúp việc đi lại thuận tiện hơn và giảm ùn tắc giờ cao điểm.
Tuy nhiên, một số ý kiến lo ngại tiến độ thi công có thể bị chậm do vướng mắc giải phóng mặt bằng."""

gold = """Sáng nay, Ủy ban Nhân dân thành phố công bố kế hoạch phát triển giao thông công cộng giai đoạn 2025–2030.
Theo đó, thành phố sẽ đầu tư hơn 5.000 tỷ đồng để mở rộng mạng lưới xe buýt và xây dựng thêm ba tuyến BRT mới.
Ngoài ra, thành phố cũng sẽ triển khai hệ thống vé điện tử tích hợp cho tất cả các loại hình vận tải.
Người dân bày tỏ kỳ vọng hệ thống mới sẽ giúp việc đi lại thuận tiện hơn và giảm ùn tắc giờ cao điểm."""

In [10]:
#tách thành từng câu
sentences = sent_tokenize(text)   
print("Số câu:", len(sentences))
for i, s in enumerate(sentences):
    print(f"Câu {i+1}: {s}")

Số câu: 10
Câu 1: Sáng nay, Ủy ban Nhân dân thành phố công bố kế hoạch phát triển giao thông công cộng giai đoạn 2025–2030.
Câu 2: Theo đó, thành phố sẽ đầu tư hơn 5.000 tỷ đồng để mở rộng mạng lưới xe buýt và xây dựng thêm ba tuyến BRT mới.
Câu 3: Đại diện Sở Giao thông cho biết mục tiêu là tăng tỷ lệ người dân sử dụng phương tiện công cộng lên 25%.
Câu 4: Ngoài ra, thành phố cũng sẽ triển khai hệ thống vé điện tử tích hợp cho tất cả các loại hình vận tải.
Câu 5: Việc áp dụng vé điện tử được kỳ vọng sẽ giúp giảm thời gian lên xuống xe và hạn chế gian lận.
Câu 6: Bên cạnh đó, thành phố đặt mục tiêu giảm 15% lượng xe cá nhân lưu thông trong khu vực trung tâm.
Câu 7: Các chuyên gia cho rằng muốn đạt mục tiêu này, cần đồng thời tăng phí đỗ xe và kiểm soát chặt xe máy cũ.
Câu 8: Thành phố cũng sẽ thí điểm làn đường riêng cho xe buýt trên ba trục chính từ quý II năm sau.
Câu 9: Người dân bày tỏ kỳ vọng hệ thống mới sẽ giúp việc đi lại thuận tiện hơn và giảm ùn tắc giờ cao điểm.
Câu 10: Tuy 

In [11]:
#lấy embedding cho toàn bộ câu
emb = get_embeddings(sentences)
print("embed shape:", emb.shape)

embed shape: (10, 768)


## K-Means

In [12]:
# Kmean
#Phân cụm -> tóm tắt
summary = kmeans_summary(sentences)
print("Bản tóm tắt dự đoán:")
print(summary)

Bản tóm tắt dự đoán:
Ngoài ra, thành phố cũng sẽ triển khai hệ thống vé điện tử tích hợp cho tất cả các loại hình vận tải. Bên cạnh đó, thành phố đặt mục tiêu giảm 15% lượng xe cá nhân lưu thông trong khu vực trung tâm. Thành phố cũng sẽ thí điểm làn đường riêng cho xe buýt trên ba trục chính từ quý II năm sau. Người dân bày tỏ kỳ vọng hệ thống mới sẽ giúp việc đi lại thuận tiện hơn và giảm ùn tắc giờ cao điểm.


In [13]:
#đánh giá
scores = evaluate(gold, summary)
print("Đánh giá:")
print(scores)

Đánh giá:
{'rouge1': 0.814, 'rouge2': 0.6007, 'rougeL': 0.5333, 'bleu': 0.5133}


## DBSCAN

In [14]:
#DBSCAN

#Phân bố khoảng cách dữ liệu

D = np.linalg.norm(emb[:, None, :] - emb[None, :, :], axis=2)
print("Min:", D[D>0].min())
print("Max:", D.max())
print("Mean:", D.mean())

Min: 3.3474774
Max: 4.834093
Mean: 3.7888608


In [15]:
#Phân cụm -> tóm tắt
summary_dbscan = dbscan_summary(sentences, eps=D.mean(), min_samples=2)    #chọn eps còn thùy thuộc vào phôn bố dữ liệu
print("Bản tóm tắt dự đoán:")
print(summary_dbscan)

Bản tóm tắt dự đoán:
Thành phố cũng sẽ thí điểm làn đường riêng cho xe buýt trên ba trục chính từ quý II năm sau. Người dân bày tỏ kỳ vọng hệ thống mới sẽ giúp việc đi lại thuận tiện hơn và giảm ùn tắc giờ cao điểm.


In [16]:
#đánh giá
scores = evaluate(gold, summary_dbscan)
print("Đánh giá:")
print(scores)

Đánh giá:
{'rouge1': 0.5926, 'rouge2': 0.4206, 'rougeL': 0.5185, 'bleu': 0.1803}


# Đánh giá trên benchmark

In [19]:

df = pd.read_json('../../data/summarization_samples.json')
df = df.dropna(subset=["text", "summary"])
df.reset_index(drop=True, inplace=True)
print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   id       10 non-null     int64
 1   text     10 non-null     str  
 2   summary  10 non-null     str  
dtypes: int64(1), str(2)
memory usage: 372.0 bytes
None
   id                                               text  \
0   1  Nghiên cứu mới đây từ Đại học Oxford cho thấy ...   
1   2  Trí tuệ nhân tạo (AI) đang tạo ra một cuộc các...   
2   3  Biến đổi khí hậu đang gây ra những tác động ng...   
3   4  Chế độ ăn Địa Trung Hải từ lâu đã được công nh...   
4   5  Kính viễn vọng không gian James Webb của NASA ...   

                                             summary  
0  Ngủ đủ 7-8 tiếng mỗi đêm giúp não bộ dọn dẹp c...  
1  AI đang cách mạng hóa ngành y tế nhờ khả năng ...  
2  Biến đổi khí hậu làm tăng nhiệt độ, tan băng, ...  
3  Chế độ ăn Địa Trung Hải ưu tiên rau củ, dầu ol...  
4  Kính viễn vọng James Webb gửi về hì

## Kết quả K-Means

In [20]:
# Chạy thử trên toàn bộ dữ liệu và tính điểm trung bình
all_scores = []

for _, row in data.iterrows():
    text = row['text']
    gold = row['summary']

    sentences = sent_tokenize(text)
    if len(sentences) == 0:
        continue

    pred = kmeans_summary(sentences)
    scores = evaluate(gold, pred)
    all_scores.append(scores)

results_df = pd.DataFrame(all_scores)
print(results_df.mean())

rouge1    0.62360
rouge2    0.32490
rougeL    0.40022
bleu      0.12539
dtype: float64


## Kết quả DBSCAN

In [21]:
# Chạy DBSCAN trên toàn bộ dữ liệu và tính điểm trung bình
all_scores_dbscan = []

for _, row in data.iterrows():
    text = row['text']
    gold = row['summary']

    sentences = sent_tokenize(text)
    if len(sentences) <= 1:
        pred = sentences[0] if sentences else ''
        scores = evaluate(gold, pred)
        all_scores_dbscan.append(scores)
        continue

    emb = get_embeddings(sentences)
    D = np.linalg.norm(emb[:, None, :] - emb[None, :, :], axis=2)
    D_nonzero = D[D > 0]
    eps = D_nonzero.mean() if len(D_nonzero) > 0 else 3.7

    pred = dbscan_summary(sentences, eps=eps, min_samples=2)
    scores = evaluate(gold, pred)
    all_scores_dbscan.append(scores)

results_dbscan = pd.DataFrame(all_scores_dbscan)
print(results_dbscan.mean())

rouge1    0.63094
rouge2    0.25007
rougeL    0.36675
bleu      0.11146
dtype: float64


# Tài liệu tham khảo

In [22]:
# https://huggingface.co/transformers/v4.3.3/model_doc/phobert.html#overview
# https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#sklearn.cluster.KMeans
# https://phamdinhkhanh.github.io/deepai-book/ch_ml/DBSCAN.html
# https://phamdinhkhanh.github.io/2020/06/04/PhoBERT_Fairseq.html#3-load-model-bert